# AWS Glue Studio Notebook
##### You are now running a AWS Glue Studio notebook; To start using your notebook you need to start an AWS Glue Interactive Session.


#### Optional: Run this cell to see available notebook commands ("magics").


In [ ]:
%help

####  Run this cell to set up and start your interactive session.


In [ ]:
%idle_timeout 2880
%glue_version 3.0
%worker_type G.1X
%number_of_workers 5

import sys
from awsglue.transforms import *
from awsglue.utils import getResolvedOptions
from pyspark.context import SparkContext
from awsglue.context import GlueContext
from awsglue.job import Job
  
sc = SparkContext.getOrCreate()
glueContext = GlueContext(sc)
spark = glueContext.spark_session
job = Job(glueContext)

#### Example: Create a DynamicFrame from a table in the AWS Glue Data Catalog and display its schema


In [ ]:
df = spark.read.option("recursiveFileLookup", "true").text('s3://cdkstack-documentsbucket9ec9deb9-sbbf9n4wdhze/embeddingarchive/')

In [ ]:
df.count()

In [ ]:
df.take(1)

In [ ]:
df.printSchema()

In [ ]:
from pyspark.sql.functions import split, col, regexp_replace, transform
df2 = df.withColumn("value", regexp_replace("value", r'(\[)', '')).withColumn("value", regexp_replace("value", r'(])', ''))
df2.head(1)

In [ ]:
from pyspark.sql.functions import split, col
df3 = df2.select(split(col("value"),",").alias("EmbedArray")) \
    .drop("value")
df3.printSchema()

In [ ]:
df3.take(1)

In [ ]:
df4 = df3.withColumn("EmbedArray", transform(col("EmbedArray"), lambda x: x.cast("float")))
df4 = df4.withColumn("EmbedArray", col("EmbedArray").cast("array<float>"))
df4.printSchema()

In [ ]:
df4.head(1)

In [ ]:
from pyspark.ml.linalg import Vectors

In [ ]:
from pyspark.ml.functions import array_to_vector

In [ ]:
df5 = df4.select(array_to_vector('EmbedArray').alias('EmbedArray'))
df5.printSchema()

In [ ]:
df5.head(1)

In [ ]:
from pyspark.ml.feature import VectorAssembler
assembler = VectorAssembler(
  inputCols=["EmbedArray"], outputCol="features"
)

dfTrain = assembler.transform(df5).drop('EmbedArray')
dfTrain.printSchema()

In [ ]:
dfTrain.head(1)

In [ ]:
from pyspark.ml.feature import PCA
pca = PCA(k=100, inputCol="features")

In [ ]:
pca.setOutputCol("pca_features")
pca_model = pca.fit(dfTrain)

In [ ]:
pca_model.setOutputCol("output")
dfPca = pca_model.transform(dfTrain)

In [ ]:
expl_var = pca_model.explainedVariance.cumsum()

In [ ]:
import numpy as np

In [ ]:
expl_95 = np.argwhere(expl_var > 0.95)[0][0]
expl_95

In [ ]:
from pyspark.ml.clustering import KMeans

In [ ]:
kmeans = KMeans(k=10)

In [ ]:
kmeans_model = kmeans.fit(dfPca)

In [ ]:
kmeans_model.setPredictionCol("newPrediction")

In [ ]:
dfKmeans = kmeans_model.transform(dfPca).select("features", "newPrediction")

In [ ]:
dfKmeans.printSchema()

In [ ]:
centers = kmeans_model.clusterCenters()

In [ ]:
len(centers)

In [ ]:
kmeans_model.summary.clusterSizes

In [ ]:
kmeans_model.summary.cluster.head(1)

In [ ]:
kmeans_model.summary.trainingCost

In [ ]:
kmeans_model.summary.predictions

In [ ]:
kmeans_model.summary.predictions.head(1)[0]['output'].shape

In [ ]:
from pyspark.ml.evaluation import ClusteringEvaluator
evaluator = ClusteringEvaluator(predictionCol='newPrediction', featuresCol='features', \
                                metricName='silhouette', distanceMeasure='squaredEuclidean')

In [ ]:
score=evaluator.evaluate(dfKmeans)
score

### Consolidated output

In [ ]:
# expl_95

In [ ]:
# centers

In [ ]:
# kmeans_model.summary.clusterSizes

In [ ]:
# kmeans_model.summary.trainingCost

In [ ]:
# score 